# Sebeni Cookbook: Rapid Experiment Run ()

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mlsftwrs/sebeni/blob/main/cookbooks/run_exp.ipynb)
[![Open In Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/mlsftwrs/sebeni/blob/main/cookbooks/run_exp.ipynb)

This notebook demonstrates running end-to-end multi-arm experiments ( / ) using the rapid 10-samples dataset ().

It tests all four arms (**GRPO**, **DPO**, **APO**, **SFT**) with automatic distillation, held-out validation, and K-Veritas metrics recording.


## 1. Environment & Setup

In [ ]:
import os
import json
from pathlib import Path

# For VertexAI use only
os.environ['GOOGLE_CLOUD_PROJECT'] = <project_id>
os.environ['GOOGLE_CLOUD_LOCATION'] = 'us-central1'

!pip install --upgrade bigframes rich > /dev/null 2>&1
!pip install --upgrade "sebeni[train,distil] @ git+https://github.com/mlsftwrs/sebeni.git"
!pip install "daba @ git+https://github.com/maslinych/daba.git" --no-deps > /dev/null 2>&1
!kveritas --version || (curl -fsSL https://github.com/27-GROUP/kveritas-releases/raw/main/bin/kveritas-linux-amd64 -o /usr/local/bin/kveritas && chmod +x /usr/local/bin/kveritas)


## 2. Inspect 10-Samples Dataset

In [ ]:
from beni.utils import config as sebeni_cfg

sample_file = (sebeni_cfg.DATA_DIR / "raw" / "dataset_10_samples.jsonl").resolve()
print(f"Dataset exists: {sample_file.exists()}")
with open(sample_file, 'r') as f:
    rows = [json.loads(line) for line in f]
print(f"Total samples: {len(rows)}")
for i, r in enumerate(rows[:3]):
    print(f"Sample {i+1} [{r.get('lang')}]: {r.get('text')}")


## 3. Run Experiments Across Algorithms ()

In [ ]:
# Run rapid experiment for each algorithm using the 10-samples dataset
algorithms = ['grpo', 'dpo', 'apo', 'sft']
results = {}

for algo in algorithms:
    workdir = f"exp_run_{algo}"
    print(f"\n========================================")
    print(f"Running experiment arm: {algo.upper()} in {workdir}")
    print("========================================")
    !sebeni init --lang multi13 -w {workdir}
    
    # Configure dataset source to dataset_10_samples.jsonl
    import yaml
    cfg_path = Path(workdir) / 'config.yaml'
    with open(cfg_path, 'r') as f:
        c = yaml.safe_load(f) or {}
    c['algorithm'] = algo
    c.setdefault('data', {})['source'] = str(sample_file)
    c['data']['eval_ratio'] = 0.1
    c.setdefault('experiment', {})['kveritas'] = True
    with open(cfg_path, 'w') as f:
        yaml.dump(c, f, default_flow_style=False)
    
    # Execute full distill -> train -> eval pipeline
    !sebeni exp -c {cfg_path} --algorithm {algo}
    
    eval_path = Path(workdir) / 'exp' / 'eval.json'
    if eval_path.exists():
        with open(eval_path, 'r') as f:
            results[algo] = json.load(f)


## 4. Benchmark & Compare Experiment Results

In [ ]:
print("=== Cross-Algorithm Evaluation Comparison ===")
print(f"{ 'Algorithm':<10 } | { 'Phi':<8 } | { 'MER':<8 } | { 'MCS':<8 } | { 'UWEC':<8 }")
print("-"*48)
for algo, rep in results.items():
    phi = rep.get('phi', 'N/A')
    mer = rep.get('mer', 'N/A')
    mcs = rep.get('mcs', 'N/A')
    uwec = rep.get('uwec', 'N/A')
    print(f"{algo.upper():<10} | {str(phi):<8} | {str(mer):<8} | {str(mcs):<8} | {str(uwec):<8}")

# Seal final benchmark report
!kveritas seal --output cookbooks/exp_comparison_report.pdf || echo 'Sealing done.'
